In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install mne loguru typer tqdm seaborn tensorflow scikit-learn

In [ ]:
%cd /content/drive/MyDrive/aid-data-analysis-smc-2025-main

#Checking log file for wrong or correct pressing button

In [ ]:
import re
from pathlib import Path

LOG_DIR = Path("/content/drive/MyDrive/data_set_org/log_files")

LOG_FILES = sorted(LOG_DIR.glob("*.log"))

print(f"Found {len(LOG_FILES)} log files.\n")

def analyze_log(filepath):
    with open(filepath, encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    events = []
    for line in lines:
        if 'Added new stimulus AttentionCheckStimulus' in line:
            events.append('AC_STIM')
        elif 'Added new stimulus Stimulus' in line:
            events.append('NORMAL_STIM')
        elif 'Attention check action conducted' in line:
            events.append('BUTTON')

    ac_with_button = 0
    ac_missed = 0        # omission error
    normal_with_button = 0  # commission error

    for idx, etype in enumerate(events):
        if etype == 'AC_STIM':
            pressed = False
            for j in range(idx + 1, len(events)):
                if events[j] == 'BUTTON':
                    pressed = True
                    break
                if events[j] in ('AC_STIM', 'NORMAL_STIM'):
                    break
            if pressed:
                ac_with_button += 1
            else:
                ac_missed += 1
        elif etype == 'NORMAL_STIM':
            pressed = False
            for j in range(idx + 1, len(events)):
                if events[j] == 'BUTTON':
                    pressed = True
                    break
                if events[j] in ('AC_STIM', 'NORMAL_STIM'):
                    break
            if pressed:
                normal_with_button += 1

    total_ac = sum(1 for e in events if e == 'AC_STIM')
    total_normal = sum(1 for e in events if e == 'NORMAL_STIM')

    return {
        'total_ac': total_ac,
        'total_normal': total_normal,
        'ac_correct': ac_with_button,
        'omission': ac_missed,
        'commission': normal_with_button,
    }


print(f"{'Subject':<10}{'AC total':<10}{'Normal':<10}{'AC correct':<12}{'OMISSION':<12}{'COMMISSION':<12}")
print("-" * 66)

sum_omission = 0
sum_commission = 0

for filepath in LOG_FILES:
    subj = filepath.stem
    r = analyze_log(filepath)
    sum_omission += r['omission']
    sum_commission += r['commission']
    print(f"{subj:<10}{r['total_ac']:<10}{r['total_normal']:<10}"
          f"{r['ac_correct']:<12}{r['omission']:<12}{r['commission']:<12}")

print("-" * 66)
print(f"{'TOTAL':<10}{'':<10}{'':<10}{'':<12}{sum_omission:<12}{sum_commission:<12}")
print()
print(f"Total error trials across all subjects: {sum_omission + sum_commission}")
print(f"  Omission (missed attention check): {sum_omission}")
print(f"  Commission (button in normal trial): {sum_commission}")

In [ ]:
import mne
import re
from pathlib import Path

INPUT_DIR = Path("/content/drive/MyDrive/data_set_p2")

def get_trigger(desc):
    m = re.search(r"Stimulus/S\s*(\d+)", desc)
    return int(m.group(1)) if m else None

OPTION_START = {50, 51, 52, 56}
TARGET_START = {70, 71, 72, 76}

raw_files = sorted(
    f for f in INPUT_DIR.glob("sub-*_ses-01-raw.fif")
    if "sub-03" not in f.name and "_2" not in f.name
)

print(f"{'Subject':<10}{'120 total':<12}{'in NORMAL (commission)':<24}{'in ATT_CHECK (correct)':<24}")
print("-" * 70)

total_commission = 0

for raw_path in raw_files:
    subj = re.search(r"(sub-\d+)", raw_path.name).group(1)
    raw = mne.io.read_raw_fif(raw_path, preload=False, verbose=False)
    events, _ = mne.events_from_annotations(raw, event_id=get_trigger, verbose=False)


    commission = 0
    correct_ac = 0

    i = 0
    codes = events[:, 2]
    n = len(codes)


    for idx in range(n):
        if codes[idx] == 120:
            has_target = False
            j = idx - 1
            while j >= 0:
                if codes[j] == 101:
                    break
                if codes[j] in TARGET_START:
                    has_target = True
                j -= 1
            if has_target:
                commission += 1
            else:
                correct_ac += 1

    n_120 = int((codes == 120).sum())
    total_commission += commission
    print(f"{subj:<10}{n_120:<12}{commission:<24}{correct_ac:<24}")
    del raw

print("-" * 70)
print(f"Total commission errors (from EEG triggers): {total_commission}")

#Building epochs

In [ ]:
from pathlib import Path
import gc
import re
import mne
import numpy as np
import pandas as pd

mne.use_log_level("warning")

INPUT_DIR = Path("/content/drive/MyDrive/data_set_p2")
ERRP_DIR  = Path("/content/drive/MyDrive/data_set_org/errp_epochs")
ERRP_DIR.mkdir(parents=True, exist_ok=True)

OPTION_START = {50, 51, 52, 56}
TARGET_START = {70, 71, 72, 76}

TMIN     = -0.2
TMAX     = 0.8
BASELINE = (-0.2, 0.0)


def get_trigger(desc):
    m = re.search(r"Stimulus/S\s*(\d+)", desc)
    return int(m.group(1)) if m else None


def process_subject(raw_path):
    subj = re.search(r"(sub-\d+)", raw_path.name).group(1)
    print(f"\n{'='*40}\nProcessing {subj}...")

    raw = None
    epochs = None
    try:
        raw = mne.io.read_raw_fif(raw_path, preload=True, verbose=False)
        raw.filter(l_freq=0.1, h_freq=20.0, verbose=False)

        events, _ = mne.events_from_annotations(raw, event_id=get_trigger, verbose=False)
        codes = events[:, 2]
        n = len(codes)

        evt_list  = []
        meta_list = []

        for idx in range(n):
            if codes[idx] == 120:
                has_target = False
                j = idx - 1
                while j >= 0:
                    if codes[j] == 101:
                        break
                    if codes[j] in TARGET_START:
                        has_target = True
                    j -= 1

                if has_target:
                    evt_list.append([events[idx, 0], 0, 1])
                    meta_list.append({"Response_Type": "error"})
                else:
                    evt_list.append([events[idx, 0], 0, 2])
                    meta_list.append({"Response_Type": "correct"})

        if not evt_list:
            print(f"  No button presses, skipping")
            return

        evt_arr = np.array(evt_list)
        meta_df = pd.DataFrame(meta_list)


        present_codes = set(evt_arr[:, 2])
        event_id = {}
        if 1 in present_codes:
            event_id["error"] = 1
        if 2 in present_codes:
            event_id["correct"] = 2

        epochs = mne.Epochs(
            raw, evt_arr,
            event_id=event_id,
            tmin=TMIN, tmax=TMAX,
            baseline=BASELINE,
            metadata=meta_df,
            picks="eeg",
            preload=True,
            reject_by_annotation=False,
            verbose=False,
        )

        n_error   = sum(epochs.metadata["Response_Type"] == "error")
        n_correct = sum(epochs.metadata["Response_Type"] == "correct")

        out_path = ERRP_DIR / f"{subj}-errp-epo.fif"
        epochs.save(out_path, overwrite=True)
        print(f"  Saved {out_path.name} — error={n_error}, correct={n_correct}")

        del epochs
        epochs = None

    except Exception as e:
        print(f"  Error on {subj}: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if raw is not None:
            del raw
        if epochs is not None:
            del epochs
        gc.collect()


raw_files = sorted(
    f for f in INPUT_DIR.glob("sub-*_ses-01-raw.fif")
    if "sub-03" not in f.name and "_2" not in f.name
)
print(f"Found {len(raw_files)} subjects.")

for raw_path in raw_files:
    process_subject(raw_path)

print("\nALL ErrP EPOCHS DONE!")

#Plot

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

mne.use_log_level("warning")

ERRP_DIR = Path("/content/drive/MyDrive/data_set_org/errp_epochs")
#ERRP_DIR = Path("/content/drive/MyDrive/data_set_org/errp_epochs_ica")

epo_files = sorted(ERRP_DIR.glob("*-errp-epo.fif"))

PICKS = ['Fz','Cz', 'Pz']

print("Loading and pooling...")

error_epochs_list   = []
correct_epochs_list = []

for filepath in epo_files:
    epochs = mne.read_epochs(filepath, preload=True, verbose=False)
    if "error" in epochs.event_id:
        error_epochs_list.append(epochs["error"])
    if "correct" in epochs.event_id:
        correct_epochs_list.append(epochs["correct"])

pooled_error   = mne.concatenate_epochs(error_epochs_list)
pooled_correct = mne.concatenate_epochs(correct_epochs_list)

print(f"Pooled error:   {len(pooled_error)}")
print(f"Pooled correct: {len(pooled_correct)}")

available = pooled_error.ch_names
use_picks = [ch for ch in PICKS if ch in available]
pick_idx  = [available.index(ch) for ch in use_picks]
print(f"Channels: {use_picks}")

ev_error   = pooled_error.average()
ev_correct = pooled_correct.average()

times = ev_error.times

# error vs correct
wave_error   = ev_error.data[pick_idx, :].mean(axis=0) * 1e6
wave_correct = ev_correct.data[pick_idx, :].mean(axis=0) * 1e6

plt.figure(figsize=(11, 6))
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=1.2, label='Button press (t=0)')
plt.plot(times, wave_error,   color='red',   linewidth=2, label=f'Error (n={len(pooled_error)})')
plt.plot(times, wave_correct, color='green', linewidth=2, label=f'Correct (n={len(pooled_correct)})')
plt.xlabel("Time (s) relative to button press")
plt.ylabel("Amplitude (µV)")
plt.title(f"ErrP: Error vs Correct — mean of {use_picks}")
plt.legend()
plt.tight_layout()
plt.show()

# ==============
# Difference wave (error - correct)
# ==============
diff_wave = wave_error - wave_correct

plt.figure(figsize=(11, 6))
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=1.2, label='Button press (t=0)')
plt.plot(times, diff_wave, color='purple', linewidth=2.5, label='Error - Correct (ErrP)')
plt.xlabel("Time (s) relative to button press")
plt.ylabel("Amplitude difference (µV)")
plt.title("ErrP Difference Wave (Error - Correct)")
plt.legend()
plt.tight_layout()
plt.show()


# Topography
ev_diff = mne.combine_evoked([ev_error, ev_correct], weights=[1, -1])
times_topo = [0.0, 0.1, 0.2, 0.3, 0.4]
fig = ev_diff.plot_topomap(times=times_topo, ch_type='eeg', show=False)
fig.suptitle("ErrP topography (Error - Correct)", y=1.05)
plt.show()

print("\nDone.")

#Plotting for inviduals

In [ ]:
import mne
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import math

mne.set_log_level("ERROR")

ERRP_DIR = Path("/content/drive/MyDrive/data_set_org/errp_epochs")
epo_files = sorted(ERRP_DIR.glob("*-errp-epo.fif"))

PICKS = ['Fz']

subjects = []
data_error   = {}
data_correct = {}
times = None
pick_idx = None

for filepath in epo_files:
    subj = '-'.join(filepath.name.split('-')[:2])
    epochs = mne.read_epochs(filepath, preload=True, verbose=False)

    if times is None:
        times = epochs.times
        available = epochs.ch_names
        use_picks = [ch for ch in PICKS if ch in available]
        pick_idx = [available.index(ch) for ch in use_picks]

    if "error" in epochs.event_id and len(epochs["error"]) > 0:
        d = epochs["error"].average().data[pick_idx, :].mean(axis=0) * 1e6
        data_error[subj] = d
    if "correct" in epochs.event_id and len(epochs["correct"]) > 0:
        d = epochs["correct"].average().data[pick_idx, :].mean(axis=0) * 1e6
        data_correct[subj] = d

    subjects.append(subj)
    del epochs

print(f"Channels: {use_picks}")

n_subs = len(subjects)
cols = 3
rows = math.ceil(n_subs / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows), sharex=True, sharey=True)
axes = axes.flatten()

for i, subj in enumerate(subjects):
    ax = axes[i]
    ax.axhline(0, color='black', linewidth=0.6)
    ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    n_err = 0
    n_cor = 0

    if subj in data_error:
        ax.plot(times, data_error[subj], color='red', linewidth=2, label='Error')
    if subj in data_correct:
        ax.plot(times, data_correct[subj], color='green', linewidth=2, label='Correct')

    fp = ERRP_DIR / f"{subj}-errp-epo.fif"
    ep = mne.read_epochs(fp, preload=False, verbose=False)
    n_err = len(ep["error"]) if "error" in ep.event_id else 0
    n_cor = len(ep["correct"]) if "correct" in ep.event_id else 0
    del ep

    ax.set_title(f"{subj} (err={n_err}, cor={n_cor})")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("µV")

for j in range(n_subs, len(axes)):
    fig.delaxes(axes[j])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=14)

plt.tight_layout()
plt.subplots_adjust(top=0.94)
plt.show()

#Building epochs for ICA

In [ ]:
from pathlib import Path
import gc
import re
import mne
import numpy as np
import pandas as pd
from mne.preprocessing import ICA

mne.use_log_level("warning")

INPUT_DIR = Path("/content/drive/MyDrive/data_set_p2")
ERRP_DIR  = Path("/content/drive/MyDrive/data_set_org/errp_epochs_ica")
ERRP_DIR.mkdir(parents=True, exist_ok=True)

OPTION_START = {50, 51, 52, 56}
TARGET_START = {70, 71, 72, 76}

TMIN     = -0.2
TMAX     = 0.8
BASELINE = (-0.2, 0.0)

RESAMPLE_ICA = 100


def get_trigger(desc):
    m = re.search(r"Stimulus/S\s*(\d+)", desc)
    return int(m.group(1)) if m else None


def process_subject(raw_path):
    subj = re.search(r"(sub-\d+)", raw_path.name).group(1)
    print(f"\n{'='*40}\nProcessing {subj}...")

    raw = None
    raw_ica_fit = None
    ica = None
    epochs = None
    try:
        raw = mne.io.read_raw_fif(raw_path, preload=True, verbose=False)
        raw.resample(RESAMPLE_ICA, verbose=False)
        gc.collect()

        raw.filter(l_freq=0.1, h_freq=20.0, verbose=False)


        raw_ica_fit = raw.copy().filter(l_freq=1.0, h_freq=None, verbose=False)
        print("  Fitting ICA...")
        ica = ICA(n_components=15, method="fastica", random_state=42, max_iter="auto")
        ica.fit(raw_ica_fit)
        del raw_ica_fit
        raw_ica_fit = None
        gc.collect()


        try:
            eog_idx, _ = ica.find_bads_eog(raw, ch_name="Fp1", verbose=False)
        except Exception as e:
            print(f"  blink detection failed: {e}")
            eog_idx = []
        print(f"  Removing blink components: {eog_idx}")

        ica.apply(raw, exclude=eog_idx, verbose=False)
        del ica
        ica = None
        gc.collect()


        events, _ = mne.events_from_annotations(raw, event_id=get_trigger, verbose=False)
        codes = events[:, 2]
        n = len(codes)

        evt_list  = []
        meta_list = []

        for idx in range(n):
            if codes[idx] == 120:
                has_target = False
                j = idx - 1
                while j >= 0:
                    if codes[j] == 101:
                        break
                    if codes[j] in TARGET_START:
                        has_target = True
                    j -= 1

                if has_target:
                    evt_list.append([events[idx, 0], 0, 1])
                    meta_list.append({"Response_Type": "error"})
                else:
                    evt_list.append([events[idx, 0], 0, 2])
                    meta_list.append({"Response_Type": "correct"})

        if not evt_list:
            print(f"  No button presses, skipping")
            return

        evt_arr = np.array(evt_list)
        meta_df = pd.DataFrame(meta_list)

        present_codes = set(evt_arr[:, 2])
        event_id = {}
        if 1 in present_codes:
            event_id["error"] = 1
        if 2 in present_codes:
            event_id["correct"] = 2

        epochs = mne.Epochs(
            raw, evt_arr,
            event_id=event_id,
            tmin=TMIN, tmax=TMAX,
            baseline=BASELINE,
            metadata=meta_df,
            picks="eeg",
            preload=True,
            reject_by_annotation=False,
            verbose=False,
        )

        n_error   = sum(epochs.metadata["Response_Type"] == "error")
        n_correct = sum(epochs.metadata["Response_Type"] == "correct")

        out_path = ERRP_DIR / f"{subj}-errp-epo.fif"
        epochs.save(out_path, overwrite=True)
        print(f"  Saved {out_path.name} — error={n_error}, correct={n_correct}")

        del epochs
        epochs = None

    except Exception as e:
        print(f"  Error on {subj}: {e}")
        import traceback
        traceback.print_exc()
    finally:
        for obj in [raw, raw_ica_fit, ica, epochs]:
            if obj is not None:
                del obj
        gc.collect()


raw_files = sorted(
    f for f in INPUT_DIR.glob("sub-*_ses-01-raw.fif")
    if "sub-03" not in f.name and "_2" not in f.name
)
print(f"Found {len(raw_files)} subjects.")

for raw_path in raw_files:
    process_subject(raw_path)

print("\nALL ErrP ICA EPOCHS DONE!")